# Linear attention, Part 3 of 3: causal attention as a recurrence

*Part 3 of 3, adapted from 2026 Round 2 Problem 1. Parts 1 and 2 are u15-p05 and u15-p06.*

The vibration model must also run online: it reads one time step at a time and may never look ahead. With a causal mask, query $i$ uses keys $j \le i$ only:

$$o_i^\top = \frac{\phi(q_i)^\top S_i}{\phi(q_i)^\top z_i}, \qquad S_i = \sum_{j \le i} \phi(k_j)\, v_j^\top \in \mathbb{R}^{\ell \times d_v}, \qquad z_i = \sum_{j \le i} \phi(k_j) \in \mathbb{R}^{\ell}.$$

Write two functions.

**1. All positions at once** (for training):

```python
def causal_linear_attention(q, k, v, feature_map):
    # Your code
    return out            # (B, H, T, d_v)
```

`q`, `k` have shape `(B, H, T, d_k)`, `v` has shape `(B, H, T, d_v)`, and `feature_map` is as in Part 2 (positive outputs of size `l`). Use prefix sums (`torch.cumsum`). As in Part 2, no intermediate tensor may have $T^2$ or more entries when $T$ is much larger than $\ell \cdot d_v$.

**2. One step of the recurrence** (for online use):

```python
def linear_attention_step(q_t, k_t, v_t, feature_map, state=None):
    # Your code
    return out_t, (S, z)
```

`q_t`, `k_t` have shape `(B, H, d_k)` and `v_t` has shape `(B, H, d_v)`: the vectors of the newest position. `state` is `None` before the first step, and afterward the pair `(S, z)` returned by the previous step, with `S` of shape `(B, H, l, d_v)` and `z` of shape `(B, H, l)`. Update the state with the newest position first, then compute that position's output `out_t` of shape `(B, H, d_v)`. Return new tensors; do not change the old `S` or `z` in place. Calling the step function for $t = 0, 1, \dots, T-1$ must reproduce `causal_linear_attention` exactly.

Neither function may modify its inputs.

**Example.**

```python
phi = lambda x: torch.where(x >= 0, x + 1, torch.exp(x.clamp(max=0)))
q = torch.tensor([[[[1.0, 0.0], [0.0, 1.0]]]])          # B = H = 1, T = 2, d_k = 2
k = torch.tensor([[[[1.0, 1.0], [-1.0, 0.0]]]])
v = torch.tensor([[[[2.0], [4.0]]]])
causal_linear_attention(q, k, v, phi)                   # tensor([[[[2.0000], [2.5659]]]])
out0, state = linear_attention_step(q[:, :, 0], k[:, :, 0], v[:, :, 0], phi)
out0                                                    # tensor([[[2.]]]): position 0 sees only itself
```

**Restrictions.** Only `import torch` and `import math`. No `softmax`, `scaled_dot_product_attention` or `nn.MultiheadAttention`.

**Constraints.** $B H \le 8$, $T \le 2048$, $d_k, d_v \le 32$, $\ell \le 64$. Time limit 1 second per call of `causal_linear_attention` on CPU.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `problem7_submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile problem7_submission.py
import math
import torch


def causal_linear_attention(q, k, v, feature_map):
    # Your code
    return out


def linear_attention_step(q_t, k_t, v_t, feature_map, state=None):
    # Your code
    return out_t, (S, z)


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import problem7_submission as submission
importlib.reload(submission)
# e.g. submission.causal_linear_attention(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9Wwtz3DaS/iu4SdWadEb0aCw7iRzlVk6UxFt+la2t7JU0xWCGmBEjDkkTpB7W6r/f1w2ABOchW/beebMaEo/uRqPfAG8GaTLYF4Nm98lOOfpuMBSDpaolmm7We+q0zhS1vUxzJSsh61rldVrkQ/FWVrV4LIq5eLwvZrLRMuu6hdRCikrNmqpS+UwRrExpXeQOQTZ6yqjTvKmVRut4hNcknc/TWZPV12jZQwPAVfQ8MBjijOmIW0QMo0iaTMW5XDKpZVVMM7X8LtbNdJlqbUfJLCsuVRKny7KoasJ4MqiLanbGIGR9NpjgYSrzXCXcqWcyw/ikqGOATJpZ3Uf7CmSmZ0omh36rLub1Ul45YHFWFCUhm8tMK+Ko0nVcp0tVNHXcrruWC0NRiyEyK2XK2zbLhKXU57w7RH40a5a6WdJ7leeMVxdNNWNeyESWtUp2xqPx05134523uzSuxNaN0V1XjbrF+zzNeAtuBmeyyrFNUck8P+X//a6yUlVa1GeyFpa5YlHJhBoNN0Xw57wqlsJOd60P/wyj0/w0P7TDRarFrMhpNY+aPK31ox/p56dHFireyzT56ZEZDSIi8aIWiZqDFcAPzol5k8+IFVrQdifcGD80tNXyXOFJnebdzgsjHOJPNP0ZiaN/Hb56+/LovchSXdOSlAFLwLTQZ8VlLjDpXZOL4CLVKagKn4n3BK4WVQO06kJV1zwJS/tVpllTYWYlU60EvYrLtD6D7AOelgslLquUtk/Mi0o8V/m+0PJaXBK5l9CRy6rIF0ORE1SRNMtSnKVJguFpXja1Ju6ZXcgtS2dFed2+kBgRf+1rjvnXpHl5Sa3fiJ+LTE7Fz2//CWWslNCkABXxKidSHmjxaizeVsVQ6EJkBcSdIKb5QotlA6ZMFTEVLCpIKVN0ZynxYYr1n2Er53IGCYwI0zga0eZKsWho/wODWQNWpnbqswpaInSpsGEYVRXN4iy7hrRkcwFaQMWsqIjRlSKm1yKdE4F4ZVW167BKQ0x5//bo6Jf418Ofj9+8EweEnRZ8ms8yCey0DcHR1UyVJCnh/mku8K9ElxkFgQLHVRJAFpOhWOqFGwO8eUFMzhPbQv+6zQ1obAckzsvgys2FnfLmuB0iDfVa51h/mutawiYGV0PTHx2rXBdV6E1ntKpuqlxcRQnM8+wsCKNZ2eAvb3MQmsGKlyleMLqjqioqD4pZsgcrLyOpZVXJa9DdrUOfyVIFi6IeQihz+gv5PDgdQLJh404HjjINXtdNmamAVo7xYcRTLS3MU4xxgwhWOBTz08ENAbzFhmuDS9xo2B11VaoZrJO48cbfAl1H2Swr9F2UDUVVF9nBrtp5OoT3MY/fO3qhWZcguSVW6voaC51nhSTCqJ1x9ju8xSzM+mhJl+Zxy3LMAuzwcNPaXBcvr0WADUk1zFtaYzboyLIgFLAUJIV+36Xt89GTJZUpW8Kc5qT5vAVuBZn2O8ssE8EMyy76Y9lFf3zBU1UFjjEjiDh4MvqRUx0sxI64DMOQYGOh6UclFFyaGJHybVCVPp/Y0IkLmcE+iCCT1YLMriKBFTf42Y8eL27D3t6rD43MPkMq+0plhhLRecKSPgyZoyACfrSy/acD3o4OyAZx8QSkG/N/Lha+aDD9seUDiFtTJp+rLQTalf1ViiEMRC0vvoMBY3yDrv+qeqTRKDT1NiPVMasIbwc96a37wQi9HemmrNFPrWCFBRxGMQdxcdwjZwHzRztnsT5wgx44qH1KY53LEp68/v81zVmRq3saZaB0UnlFIsm+fRDCJSasvX16giSdgZEUuIQ+Pf3F9agjgEGiVHlwjEjPEzBL4DHYt0reOoCw50KoKSKY3HfVYzy2JZBDMf1yvsvP4rsZYvSC8X0p2yWxneHovimwiKRBooNpuMnWYHpnZzZOb+3E1D7y3vY1m1YwNNYOkp37e+XgMAQveCniZQOlQHQTzJGKPYQ5hWY8fHh+SU+OEhM6/gwPIOY5Y55TgEprQIyDKHAB9ZI5QsZq0SwRlMOFiDKTMxW5qJO9PukT7OJJp1ky5HhW0gTCODEjEZ2R88iDVYJMN01JIchYsA5pqkIooyoJ//YxLQMzh3H0BJztSStc1ohcI8WBLCaKAvx0nlJkSSFzt5ZvblLxrdi9fSYui+qconrJwito+xCPtvbSMhnEdxymID7NA60oFARVn8vlZz6LYV39kHtDJM05KfyhSTWZQRzrPmhnIng3ORShegMQFaVgSJCWmpbEcfgrOUMCnvMULNJE3ZStcGBPwQGyi51pk8CSUjPH4AjLU0qgmhLqJOrLdKYIIDiUwkMXc6vBoNlxVLMIUcsc8RK5b4KAGQ36InFMHWmFZiwme0AoyC8YZMRhlfTEqh5RNAkIEbLLecxgVOWszSdEKak3T0aMUttwxK73QNhNFI+EnzZ0IhmTKFakDcHYlztsItD8eGAhrZiiKXKa865JjofifAxsffvIJA9XGnsr+TQrmE5ixJjZMPaamQ3I2IKEXOwWdvjxpr8g1qO6KFhukPwn9X40nt8iyoeUqWc262qFmIUJaseARJNTPn9jWWPncQ6JxM5t8YZ/jBKZJdLgOURrQQGsSTJv7D4xINKOOxW0yhfQTpW4DesSHOxkUizB7bmE+MXtwFMuzzRayrSIucLQq3L8Ri3QAFDWFQ9SNhpnaZaQ1s2oMoKEOSVdoJH/eP/mtaA6DWmOrhNQyLpaXtdnaNl5IRwe8aMtccRJWv0kfuxwoBPv0Mp/c1v901rKD2Xz0n9s7FWdpdO2yfygJWrqNOuai/bxL110RYJS1mf+dJ0uctlN09d6pbzgXir4hamcnbctkOKcigWm1qBnKZaZq9bcoeE8rXcyhWFiN3oa7dqYTu9TnWhJ5cAhmxRkF9b5OONXyhRcXpzmDkc0TzNItHsNTgcgu6gUZYC20oJINHr4z/w8pxqOLjKqqBRcAQDCtKRdix7aSPH3o3dH0BzLi+gtfoM4pjJYHIcRDBdNR1RXygqUnebgSkSDI0irqupgBGVA+EBgQgDcVPviWpD4Br76g9wXR3ujMVfCXv5x+D/v4+eHr18f/QIKbk4Hsa1IxjEtBgtz+2le1ZWa2SfE+eZpVixL0Ho6uD3NX7558zZ+/eaXo/cAF0BWol8L6Co9/AHBVebxUF/ns7bjJWLInwHDvL1X3ssvCDK7t99UTv65qI6uytAvsRybWsynqiyxRKa3DEjGmiXsDdyWajWWU0UHyK9GUGQzi2dnanauA13NaItr2fe374y+UzRMJWjwsTg3zFnqBRjzDGaRIkBT42O10fWDdnc4GLKlRHJUdZWasmLPRRkndDLZFM3WlUJgx1yClCC/BqH9OPT9NfLzK45DqSin1uPDE0v2PqSFq8OW+n0yk5qn2xS5yI2duVFcF86L2316xmgkP7eWQFvhJpcAfkUwzNCTlbI38ehkYgmdyrad/WTtXA6XQ+NGMzCvfQp9iE2t1LaTiCNkhO6mCRWpEYlVJlAwgAHBjA8oo6GgO51SwR/LOefYTWURCbZFwNVyj+dLWS+bzG+Zc3UEpppsM5h/KbPzgDZjxXV7YToNN/JsUoPVrKINZWlgxOTurzswW1JBvBxIHkRB+ZJKC+Y1AjYwkiLHYInIE+aIkjkCvmRizUaE+5t9o7cXkUwSiyNcH9ztjTcu0mWW0nYTzpOd3cmGid3mRU2ZUNS9Ya6fJGZ3s/FXWL3V1WBDsVfMR1t4x/I7hdrAS55hGYnnr+DiKgcBbYULHut6HNgkAStTtzHvi6byOu/L9EOnO6trXxEIpoMU7bOgvvYs8l0A02T7qjYP2oSxc1arOFnxI1mWKk8CV/Exk04HxuQZ8/7ffX5twsJe8Hmav7Fe7LBZHGryQaGpraxMiAo77pWs6UxvTaaNDboncSDME0gPJAWJFtRW8++iAKMlRuSplRWCxkV/FZSesoENPDxhuNm6fEseBYxxoG4eDMUDA8RpFCngA+BD3rt4QKXY2831xE30U8liA/na0d4B+6Y9QZzKXBNvZ2fWQ3Ao2DkILj8riST2X/wbPjPZaHYpr/WOOadFJoO4MG0BkVBaaEWeXQ8dTmQunasJjBTYw1MTSlHQcS0ytUBqfklhk+CRURSFnjNDUCaSgjbFAD5LOb80uxB4TvNv7Bw9L2zItc4XYvhvEXi+9G+iFxd6IgQM9xYdUKCF5Q/BD3S4QXYAOfyaHc4Lh6M9ku12GZT3Fx8bJYqL0sST/WobDTdKtqJ6n7NgyighF38XDjalrJ7eiMBEThbDyWjiC/eG5d+HBR3WuxbPxm1tydz6ZQs2YRILe2+haCY1JqGu1JnKOYltY0ee9Z9dv13Z7R2Z+rxKASa7jjmSDQACiW5M2VQ/on+Tqx2mM1F6VqWcVYgSogr9Xgh7Es0DUnd8TQkb1FXROM2H9f0K05TKKi5vjZA347mO62mgojhuO5Dy2WCTgCPSnLNTnxMegMBOzSn9VC7mo8yvXYKNSi+pZEIFK7KyXJu0Qoe/FI55UXuvibaCBYZG83FaR75lJp2Y8AmJ8s5HKPq/vWGstzTeMXuJSDxoczFY2tUslzJZWS0uTtoQ0S1l68jxxEuJLacoAD8Q7ZDHLlyH3KOdCg5RVshEBwET8YjEhnSCSxEDggefRZWMwBk7pE8E0RLjD3Dc4GrigVjXC5MwUtPJhN4p1dPdqynMGJG1mTRJIjW8LrAFPvyTDtqEyNmSiraKThEsqaA+MVRN2FvQG2vJxA9nXUrn4lnNdcc1tL6V4KpFwOykyyE6MMN9L2+kxJU85yA6LaL3NZVPXrxxzDNFnsj8BPbt/YvfDl++ezW0WbodCnMWc3HXnQJ7tqx/d4l4OR45WvrZMV+C6YpV2M4krRSUz2xGAEpXwyxHo6pTQlE5Ml8cv3h19C5+d3T4cthStxLm6FKR9PRLYRG1xlSbMZUdqk7ykYm3Iu/qmBGNrjCHbfJM1Qq+ZrqOzsJihIQ6oD+rE6ExZpw+oW5OGCZG8NeXxDqkqohqQLGZRpYn/ALGjfq1CVt26ZUkrCBa5ZgYa2bWSBaYrRpb3LoozvlMQdy4/diPRlQBdqbLYnkutWorRKYG0pXEnr88Go12OwLutYxtFNuDISaTTmLohlhhN8oGQXf4o1at10BvYpSzK4QX8kwSxSfxARLwvdFotD/5YkXmW4/9Og43+fc7zBh3VuxONLGWoekKP725Ha+SQpmgyVy2g3chELd+3n7PJSzup5OnA3Ppj7SwdRntPcB23fYe4ad1b+Ep32KbLhlodpS97mcLWy7LM0PIMrirgy6It2aPfA0FDQESlbkx7PREtv1CVtrhiNJaLXXAt2fyvldgq+rO/ekARYKMYB76BS9YCTqE5JCEMK4Ez+yOSfqrBnbLCAVFK3yLIHdLWzG4ZJ7TvFH+/s2MgzX2cJ+BnDzZn8B+87kwqLVVasHm8XRgQbvBHjbqpWowpT2b0hRrb82hDzWNotHtfc7E1i8+fIHX+VrPYw/n1q3yF5k0uwknHedIUynwXr+7sWbBu8km9jA6zsY6K/KFO5dZtdkiSHNzy4yj+SElEHNZCXcyGPpGwCLnw42VmvZXLrilmcLrDXdV7uVL/lO0+PcM+Lwi+Uwf0sFqBZzgVZDiJNh2UDsUjze5NxvOTlxCVrWG7d6uqJ1gYtUDSh2DLna9KyplozItiixYI8z0Eai6J7kEru6Ba2e4BGWrS+FrzXPhEh5j3eKY8hs6KNt3xwOU7piT3dZZ0Jlu/64Rn8+Zeg95Cx3F5oQ2TlJdcuHIjj+mMb/YxlfY9e1ne0M+Bx+aG6pDYW8BejdzhvYSCa2kvXYOR2ENvrqSy5IM58Q/V3tpLkaa20/BGjmrZ2CzokrMCVfeLKd04GniAnOFw2TM9qplzSD5ug9zwhYuOMnmrxtU/+yLj+9isgtxHGiV9SynbkqS2qgd4MktjY0c1gMxcpdXDEDG3fLdgh5yDam9V0i3Iw7orqu5KHFAWVrYr0+RbcOU7naIvVRBxuvm1o9JSAgLEsKApvXLxWgZioBvYrqLdSYJP0HPZNVVrMy9+6baBk7QXVq/CZaW7nKrLFiPofollPIsjVXWdBcae/fguAoQXImfwOyhuKJLT442dVUGVxFEa1kGQH+AbM07aK3UXPHXOcEH8HooLoaEyeGYo3F+TlWBszT4EMHAUFgShjwoOO8azPBjjPxgLrmd7IytvfmDzqTnH8TfASmqK5nrkq4i74yHYmcXzH5o6YTEwnjwI/ZaB8dDgf8SkocD08y56NO9sF9oCv4A6IuOFESNf0S6WQZJujzYARvOlSrp2d6pa1cOgxU8tDq74RqJwUmj8nbYwp2FW4ras3HowVLmdHtP8wV4AudfYafigYnLWzYjxFPttWsjzv6dPgyOtnzv5AGxUuFSfL7DzxJ94d9JZlS3+4SlbLo7NeZOOA/fLAYMejOI7tL9Xnfp/sldl3WaHCGYgoG/gw3kakAzfeEBdpDOD3uHvnV3TcvJ2XiykuSYRDpaZRojDz6c7A8F/qsnwO89X3jPduFDQ83qlWxP/SsyVGw5jPfLVE5tITmqcZ9tG6kxX9g4MaZ6Cpmi90PxMQx7d8FtV64uY6aJogiltxPWDlwjr+3ZRCS5CgPffvojJN/CsTT1SErndqgttNNerR4hElX+NV2ecDICi4sswa9/BuOP2LUjdifhJgKJce6+J9+nzBJHi7262qO0VQpiYbfPW/QjrvuTrUBmfKTdsbVjJNbhLn57OwTaXWt/H9sjRMYVbrA4mA9bQUNh/WC2xqGVxE6X/OCB0w7L+Q+ULhljaoKHE/zbjeARkFeBqScjekbDBP8s7vOtc3bNnJ0WQDfpYtOksRm/1xu5yfLRhSYb+HQ3PYciJrHeYiXaKUMjANNr/r23KdsCx9k1fYdh6zF/SpGLSmLz2YW/B96C2cNAxx6Dl/jZM27mYDfc2jVe73piux4z9cHzofgdjjEkp+oGoQeUoOVJ7yMIJtS4kLS+7smJYQzxm5DBRwLU7qijY8/Rsd71ZL3rse162ru3e4cDC7b4r3MAvKALtOedSl30P6s4H1sNfro/Ed/az/5YJP2OnQPx2HWAmvFnkMOoVwlqxcoB3386Yffkv54OWibvmyv1VCpFkJOaj1SfUvnxh/ayvZU0MaLmJyRwTsqerm4ea0LMx9pG4u7YxDFvxw/dRn3nyVK/5/u1HreDP/j6aKz+nUr5lbp4XxXszDlIs9TvUca8DmyeIs8X71uizLSP3bStcz62c1zky3H3WrBrlvmRAuQ23BxvhyrAvCUlZBw5x3+Ri/88W4Oo8DieXuMPJKFKr3whoIh7dzTeu0OvEUR/7yzPqK+8ftfu9i5325xLa/3sNKQqzCWJ5lpu9gX6zxEDQ2tTph+xxIdEDfw0LfbmGH7a1Kzd1c1ZpSTdMZQuv2U6b3pwbl1KTHxGJCQB+hh5kgFJGG6fiQZZXwmRTa9A/FJ/SZzckvjJOLm3xVS3SOnzis0KHriN2WV/4LRVh61MRhUCqBRBIYOKA851ulpOsEuOAhTsPgm/1kjzJtHdgB7KoS3V8ypMpGZY4B8rUOBjvlhR9uN2PPKX4EI2NX3R0n2Rw6m6yw2lifk5EtmjPxvTQlbDMKLz/EtZJc5lfKiwEPz/gs4Mgnrte73tvON0w/HBMS7wZKCF3DHpP0MzYfe+enVl9RMi5gMfRXzg4ntwbig4HZyb9wtDzengAu+TDZ+ARrRIP2pnufW2Li+Ee4O4SwpzxA0R0P8+tf0qOzLbT4Taxz48FpbwUeJgbPlce9X1lZxH3+Xu9ujoeQ9W6vHYWaqnnmfb0P3d3d3f91TDfv7F+/eVyfgKPnYSbJjETGrVy8sHt7f/C2qnTT0=")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
